In [68]:
import os
import pandas as pd
from PIL import Image

# 개별 문자 이미지 디렉토리
label_dirs = {
    "D": "data_d",
    "E": "data_e",
    "F": "data_f",
    "N": "data_n",
    "O": "data_o"
}

entries = []

# D, E, F, N, O 단일 문자 추가
for label, folder in label_dirs.items():
    if not os.path.exists(folder):
        continue
    for filename in sorted(os.listdir(folder)):
        if filename.endswith('.png'):
            entries.append({
                "filename": os.path.join(folder, filename),
                "label": label
            })

# "OFF" 합성 이미지 생성
output_dir = "data_off"
os.makedirs(output_dir, exist_ok=True)

off_count = 50  # 만들고 싶은 개수

for i in range(off_count):
    o_path = os.path.join("data_o", f"{i}.png")
    f1_path = os.path.join("data_f", f"{i}.png")
    f2_path = os.path.join("data_f", f"{i+1}.png")

    if not (os.path.exists(o_path) and os.path.exists(f1_path) and os.path.exists(f2_path)):
        continue

    try:
        imgs = [Image.open(p).convert('L') for p in [o_path, f1_path, f2_path]]
        widths, heights = zip(*(img.size for img in imgs))
        new_img = Image.new('L', (sum(widths), max(heights)), color=255)

        x_offset = 0
        for img in imgs:
            new_img.paste(img, (x_offset, 0))
            x_offset += img.width

        save_path = f"{output_dir}/off_{i}.png"
        new_img.save(save_path)
        entries.append({"filename": save_path, "label": "OFF"})
    except Exception as e:
        print(f"❌ OFF image 생성 실패 {i}: {e}")

# 최종 CSV 저장
df = pd.DataFrame(entries)
df.to_csv("labels.csv", index=False)
print(f"✅ labels.csv 생성 완료! 총 {len(entries)}개")


✅ labels.csv 생성 완료! 총 2415개


In [80]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# 문자셋
charset = ['D', 'E', 'N','O','F','OFF']
num_classes = len(charset)

# 전처리
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Dataset
class SimpleOCRDataset(Dataset):
    def __init__(self, csv_path, img_dir, charset, transform=None):
        self.data = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform
        self.charset = charset
        self.char2idx = {c: i for i, c in enumerate(charset)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(self.img_dir, row['filename'])
        img = Image.open(img_path).convert('L')
        if self.transform:
            img = self.transform(img)
        label = self.char2idx[row['label']]
        return img, torch.tensor(label, dtype=torch.long)

# 모델
class SimpleCNNClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# 설정
csv_path = "labels.csv"
img_dir = ""
dataset = SimpleOCRDataset(csv_path, img_dir, charset, transform)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNNClassifier(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 학습
for epoch in range(20):
    model.train()
    total = 0
    correct = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Acc: {acc:.2f}%")

# 저장
torch.save(model.state_dict(), "softmax_mixed_model.pth")


Epoch 1, Loss: 0.1716, Acc: 85.18%
Epoch 2, Loss: 0.0664, Acc: 95.24%
Epoch 3, Loss: 0.0018, Acc: 96.85%
Epoch 4, Loss: 0.0215, Acc: 97.93%
Epoch 5, Loss: 0.0032, Acc: 98.18%
Epoch 6, Loss: 0.0026, Acc: 98.63%
Epoch 7, Loss: 0.0178, Acc: 98.43%
Epoch 8, Loss: 0.0000, Acc: 98.84%
Epoch 9, Loss: 0.0002, Acc: 98.84%
Epoch 10, Loss: 0.0004, Acc: 99.21%
Epoch 11, Loss: 0.0001, Acc: 99.30%
Epoch 12, Loss: 0.0040, Acc: 98.47%
Epoch 13, Loss: 0.0001, Acc: 99.09%
Epoch 14, Loss: 0.0003, Acc: 99.54%
Epoch 15, Loss: 0.0001, Acc: 99.50%
Epoch 16, Loss: 0.0009, Acc: 99.46%
Epoch 17, Loss: 0.0000, Acc: 99.54%
Epoch 18, Loss: 0.0003, Acc: 99.71%
Epoch 19, Loss: 0.0002, Acc: 99.79%
Epoch 20, Loss: 0.0002, Acc: 99.75%


In [87]:
def predict_softmax(model, image_path, device, charset):
    model.eval()
    image = Image.open(image_path).convert('L')
    image = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(image)
        pred_idx = output.argmax(dim=1).item()
        return charset[pred_idx]

model = SimpleCNNClassifier(num_classes)
model.load_state_dict(torch.load("softmax_mixed_model.pth", map_location=device))
model.to(device)

print(predict_softmax(model, "sample_data/off_14.png", device, charset))  # → 'OFF'


OFF
